# 01 — Exploratory Data Analysis

This notebook loads the JSE Top 40 and macro data, inspects data quality, and explores the feature distributions that will drive regime detection.

**Run order:** This is the first notebook. Run `src/data/loader.py` first if you haven't downloaded data yet.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.data.loader import get_price_data, load_config
from src.data.features import build_features, get_model_input, save_features
from src.utils.plotting import RegimePlotter

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 5)
print("Imports OK")

## 1. Load Data

In [ ]:
config = load_config('../config/settings.yaml')
prices = get_price_data(config, source='yfinance')
print(f"Shape: {prices.shape}")
print(f"Date range: {prices.index[0].date()} → {prices.index[-1].date()}")
prices.tail(5)

## 2. Data Quality Check

In [ ]:
# Missing values
print("Missing values per series:")
print(prices.isnull().sum())
print()

# Basic statistics
print("Descriptive statistics:")
prices.describe().round(2)

In [ ]:
# Plot all price series normalised to 100
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(prices.columns):
    norm = prices[col] / prices[col].iloc[0] * 100
    axes[i].plot(norm.index, norm.values, lw=1.2)
    axes[i].set_title(col.upper(), fontweight='bold')
    axes[i].set_ylabel('Indexed (Base=100)')
    axes[i].grid(True, alpha=0.3)

if len(prices.columns) < len(axes):
    axes[-1].set_visible(False)

fig.suptitle('All Series — Normalised to 100 at Start', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Returns & Volatility

In [ ]:
returns = np.log(prices / prices.shift(1)).dropna()

# Rolling 21-day realised vol for JSE Top 40
rv_21 = returns['index'].rolling(21).std() * np.sqrt(252)
rv_63 = returns['index'].rolling(63).std() * np.sqrt(252)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

# Price
ax1.plot(prices['index'].index, prices['index'].values, color='#2c3e50', lw=1.2)
ax1.set_ylabel('JSE Top 40 Level')
ax1.set_title('JSE Top 40 — Price Level', fontweight='bold')
ax1.grid(True, alpha=0.3)

# Volatility
ax2.fill_between(rv_21.index, rv_21.values * 100, 0, alpha=0.5, color='#3498db', label='21-day RV')
ax2.plot(rv_63.index, rv_63.values * 100, color='#e74c3c', lw=1.5, label='63-day RV')
ax2.set_ylabel('Annualised Vol (%)')
ax2.set_title('Realised Volatility', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Avg realised vol: {rv_21.mean():.1%}")
print(f"Vol during COVID spike: {rv_21['2020-03':'2020-05'].max():.1%}")

## 4. SA-Specific Regime Indicators

In [ ]:
# USDZAR vs JSE — key SA relationship
fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

# Rolling correlation
rolling_corr = returns['index'].rolling(63).corr(returns['usdzar'])
axes[0].fill_between(rolling_corr.index, rolling_corr.values, 0,
                     where=rolling_corr.values < 0, color='#e74c3c', alpha=0.5, label='Negative (typical)')
axes[0].fill_between(rolling_corr.index, rolling_corr.values, 0,
                     where=rolling_corr.values >= 0, color='#2ecc71', alpha=0.5, label='Positive')
axes[0].axhline(0, color='black', lw=0.8)
axes[0].set_title('63-Day Rolling Correlation: JSE vs USDZAR', fontweight='bold')
axes[0].set_ylabel('Correlation')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# VIX
axes[1].fill_between(prices['vix'].index, prices['vix'].values, 0,
                     alpha=0.5, color='#9b59b6')
axes[1].axhline(30, color='red', ls='--', lw=0.8, label='Stress threshold (30)')
axes[1].set_title('VIX — Global Risk Sentiment', fontweight='bold')
axes[1].set_ylabel('VIX Level')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Build Features

In [ ]:
features = build_features(prices, config)
print(f"Feature matrix: {features.shape}")
print(f"Date range: {features.index[0].date()} → {features.index[-1].date()}")
print()
print("Columns:")
for col in features.columns:
    print(f"  {col}")

In [ ]:
# Feature correlation heatmap
plotter = RegimePlotter(config)
plotter.cfg['paths']['figures'] = '../results/figures/'
_ = plotter.feature_correlation_heatmap(features, save=True)

In [ ]:
# Save features for use in subsequent notebooks
save_features(features, config, 'features.csv')
print("Features saved.")